# Lab 3: Basic Back Propogation

#### 1) Introduction
In this lab we will use back propagation to forgo the idea of creating the individual gates and train our network to behave as a XOR gate directly.

In [1]:
# Importing libraries:
import numpy as np, random , os , math
lr = .5  # learning rate

bias = random.random()   # output layer bias
bias2 = random.random()  # hidden layer bias

weights = [
    [random.random(), random.random()],  # hidden neuron 1
    [random.random(), random.random()],  # hidden neuron 2
    [random.random(), random.random()],  # hidden neuron 3
    [random.random(), random.random()],  # hidden neuron 4
    [random.random(), random.random()],  # hidden neuron 5
    [random.random(), random.random(), random.random(), random.random(), random.random()]  # output neuron
]

nHN = len(weights)-1
print("Number of hidden neurons:", nHN)
print("\nWeights:")
for i, w in enumerate(weights):
    print(f"  Neuron {i+1}: {w}")
print("\nHidden layer bias (bias2):", bias2)
print("Output layer bias (bias):", bias)

Number of hidden neurons: 5

Weights:
  Neuron 1: [0.7862804808040238, 0.9196341695096019]
  Neuron 2: [0.13691504190353243, 0.24632409337511274]
  Neuron 3: [0.4251249857377415, 0.9996374168201428]
  Neuron 4: [0.00809059991779193, 0.6617133890461356]
  Neuron 5: [0.5944528829078743, 0.3472944411284704]
  Neuron 6: [0.65603439682101, 0.7220981763021982, 0.22154770358463938, 0.5233417479803096, 0.4948009724935266]

Hidden layer bias (bias2): 0.7801608277768471
Output layer bias (bias): 0.6392687669690645


Here I have repurposed our perceptron from lab 2 to be used as a neuron in a hidden layer. Perceptron 2 is is the output layer neuron calulation. This is very similar to the summation function back in Lab 2 but this time with addition instead of multiplication.

In [2]:
def sigmoid(x):
    # print("why is that happening:" + str(x))
    # print("why is that happening negative edition:" + str(-x))
    return  1/(1+math.exp(-x))

# This is a neuron in the hidden layer
def Perceptron(input1, input2, w, b):
    return  input1*w[0]+input2*w[1]+b

# This is a neuron in the output layer
def Perceptron2(inputs, w, b):
    outputP = 0
    for k in range(0,len(weights[len(weights)-1])):
        temp = inputs[k]* w[k]
        outputP += temp
    return sigmoid(outputP + b)

This is where the lab officially begins. There is a piece of information that we need from this for later. I think with context clues it's pretty easy to figure out. This information WILL be needed during back propogation

In [3]:
# Inputs should be of form [x,y]
def feed_forward(inputs):
    O = []
    I = []
    for k in range(0,nHN):
        temp=sigmoid(Perceptron(inputs[0],inputs[1],weights[k],bias2))
        I.append(temp)  # Store the hidden neuron output
        
    outPut = Perceptron2(I, weights[nHN], bias)
    return outPut, I

out, hidden = feed_forward([0, 1])
print("Hidden activations:")
for i in hidden:
    print(f"{i:.4f}")
print(f"\nFinal output: \n{out:.4f}")

Hidden activations:
0.8455
0.7362
0.8557
0.8087
0.7554

Final output: 
0.9377


I have no real concern with how you do the following section it's pretty difficult to do especially starting out. I have some equations for my implementation. If you don't like my implementation and want to use some other implementation go right for it as long as you can show that you train your model and end up with the correct answer.(hint maybe use dot product?)

In [10]:
def train(inputs, target):
    global weights, bias, bias2

    # ---- Forward pass ----
    output, hidden = feed_forward(inputs)

    # 1. Output neuron deltas
    # ∂E/∂yⱼ = (tⱼ - yⱼ)
    error_output = target - output
    # dyⱼ/dzⱼ = yⱼ * (1 - yⱼ)
    derivative_output = output * (1 - output)
    # ∂E/∂zⱼ = ∂E/∂y_j * dy_j/dz_j
    delta_output = error_output * derivative_output

    # 2. Update output neuron weights
    # Update output bias (bias ← bias + α * ∂E/∂zⱼ)
    bias += lr * delta_output
    
    # ∂Eⱼ/∂wᵢⱼ = ∂E/∂zⱼ * ∂zⱼ/∂wᵢⱼ = ∂E/∂zⱼ * hᵢ
    for j in range(len(hidden)):
        weights[-1][j] += lr * delta_output * hidden[j]  # "Δw = α * ∂E_j/∂w_i"

    # 3. Hidden neuron deltas
    hidden_deltas = []
    for j in range(len(hidden)):
        # dE/dy_j = Σ ∂E/∂z_j * w_ij  (here just one output, so no sum)
        dE_dy = delta_output * weights[-1][j]
        # ∂z_j/∂ = h_j * (1 - h_j)
        derivative_hidden = hidden[j] * (1 - hidden[j])
        # ∂E/∂z_j = dE/dy_j * ∂z_j/∂
        delta_hidden = dE_dy * derivative_hidden

        hidden_deltas.append(delta_hidden)

        # Update hidden bias (bias2 ← bias2 + α * ∂E/∂z_j)
        bias2 += lr * delta_hidden

        # ---- Update hidden weights (Δw = α * ∂E/∂z_j * input)
        weights[j][0] += lr * delta_hidden * inputs[0]
        weights[j][1] += lr * delta_hidden * inputs[1]

    return output

I use the following function to calculate total error in order to show measure the model's performance. It measures total error of the model.

In [11]:
def calculate_total_error(training_sets):
    total_error = 0
    for t in range(len(training_sets)):
        training_inputs, training_outputs = training_sets[t]
        out, hidden = feed_forward(training_inputs)  # You still need to figure it out!!!!!!!!!
        for o in range(len(training_outputs)):
            total_error += 0.5 * (training_outputs[0] - out) ** 2
            
    return total_error

You need to use the above function or one similar to show that your model is learning. Note that here it is constantly decreaseing.Here in the for loop I run 10000 iterations of the train function. Feel free to change it.

In [17]:
training_sets = [
     [[0, 0], [0]],
     [[0, 1], [1]],
     [[1, 0], [1]],
     [[1, 1], [0]]
]

for y in range(0,500):  # for loop
    inp, out= random.choice(training_sets)
    #print(inp,out)
    train(inp,out[0])
    error = calculate_total_error(training_sets)
    print(str(y)+str( error)) 

00.001105323718796181
10.0011051833300270387
20.0011051012546873674
30.001104973239773503
40.0011048469963330341
50.0011047291597313185
60.0011047679774411647
70.001104683378604397
80.0011045642129256929
90.00110444720040832
100.0011043385115477064
110.0011042478804600764
120.0011041450726455614
130.0011040530467917072
140.001103960799436103
150.0011038762827859348
160.0011037806570229753
170.001103685975131222
180.0011036881150819247
190.0011037109132105477
200.001103753815842204
210.0011036342276640318
220.0011036848252985298
230.0011035604720060204
240.0011034763141739511
250.001103351417700085
260.0011032406207999543
270.0011031320021581135
280.0011030418301335697
290.0011029525940358463
300.0011028474848295436
310.0011027505926384927
320.001102661848661014
330.0011026559804905643
340.0011025622375514083
350.0011024766000141163
360.0011023802758271723
370.001102284893000707
380.0011022816568666677
390.0011021906843179685
400.0011021074150137262
410.0011020275598428008
420.001101959

https://mattmazur.com/2015/03/17/a-step-by-step-backpropagation-example/ has a consistent setup on git hub. You can base your submission on it but don't copy paste!

In [18]:
val = [1,0]
for x in val:
    for y in val:
        out,I = feed_forward([x,y])
        print(out)
        print(x,"or",y,"is: ",round(out)," : ", out)
error = calculate_total_error(training_sets)

0.02037719986602679
1 or 1 is:  0  :  0.02037719986602679
0.9784747296775985
1 or 0 is:  1  :  0.9784747296775985
0.9783382041536539
0 or 1 is:  1  :  0.9783382041536539
0.028175517700580115
0 or 0 is:  0  :  0.028175517700580115
